# Weather Prediction Results & Comparison

This notebook analyzes the performance of various weather prediction models. 

**Objective:** Identify the best model for precipitation forecasting across 1, 3, and 7-day horizons.

## Model Configurations

The following models were trained and evaluated. Note the difference in splitting strategies (Temporal vs. Sequential/Spatial).

### 1. Baselines (Reference)
* **Persistence:** Assumes tomorrow's weather is the same as today's. 
* **MA-3 (Moving Average):** Predicts using the average of the last 3 days.
* *Note:* We calculated these always within specific locations.

### 2. Tree Global (Gradient Boosting)
* **Library:** `sklearn.ensemble.GradientBoostingRegressor`
* **Training Split:** 80/20 Sequential Split (on concatenated data).
* **Key Params:** `n_estimators=300`, `learning_rate=0.05`, `max_depth=3`.
* **Features:** Lagged precipitation + Static attributes (Area, Elevation, Slope, Forest Fraction).

### 3. LightGBM Global
* **Library:** `lightgbm.LGBMRegressor`
* **Training Split:** 80/20 Sequential Split.
* **Key Params:** `n_estimators=500`, `learning_rate=0.05`, `num_leaves=31`.
* **Pros:** Significantly faster training than standard trees; uses `subsample=0.8` and `colsample_bytree=0.8` to prevent overfitting.

### 4. LSTM (Deep Learning)
* **Library:** `Darts` (BlockRNNModel)
* **Architecture:** RNN with LSTM cells.
* **Training Split:** **Temporal Split** (First 80% of *each* basin is Train, last 20% is Validation).
* **Input/Output:** Lookback window of **30 days** to predict **7 days** ahead.
* **Key Params:** `n_epochs=15`, `hidden_dim=20`, `n_rnn_layers=1`, `dropout=0.1`.
* **Loss Function:** L1 Loss (MAE) - less sensitive to outliers than MSE.

### 6. TFT (Temporal Fusion Transformer)
* **Library:** `Darts` (TFTModel)
* **Architecture:** Attention-based Transformer designed for multi-horizon forecasting.
* **Training Split:** Temporal Split (80/20).
* **Key Params:** `n_epochs=5`, `hidden_size=32`, `num_attention_heads=4`.
* **Pros:** Interpretable attention weights; handles static covariates natively.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.colors as mcolors

sns.set_theme(style="whitegrid")

RESULTS_DIR = Path("prediction_results/")

# All-model comparison (baselines + ML + NN)
MODEL_FILES = {
    "Baseline MA-3": "baseline/baseline_metrics_ma-3.csv",
    "Baseline Persistence": "baseline/baseline_metrics_persistence.csv",
    "Tree": "tree_global/metrics_tree_global.csv",
    "LightGBM": "lightgbm_global/metrics_lightgbm_global.csv",
    "LSTM": "lstm/nn_lstm_global_metrics_per_day.csv",
    "TFT": "tft/tft_global_metrics_per_day.csv",
}

# Best-model comparison (for tuning impact)
BEST_MODELS = {
    "LSTM": "lstm/nn_lstm_global_metrics_per_day.csv",
    "Tuned_LSTM": "lstm/nn_lstm_global_best_metrics_per_day.csv",
}

HORIZON_ORDER = ["1 Day", "3 Day", "7 Day"]

def normalize_horizon(target: str) -> str:
    t = str(target).lower()
    if "1d" in t: return "1 Day"
    if "3d" in t: return "3 Day"
    if "7d" in t: return "7 Day"
    return t

def _aggregate_if_per_location(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """Baselines are sometimes stored per-location; aggregate to one global score per horizon."""
    if "location" not in df.columns:
        return df

    rows = []
    for target in df["target"].unique():
        sub = df[df["target"] == target]
        if "n" in sub.columns:
            total_n = sub["n"].sum()
            rmse = np.sqrt((sub["RMSE"]**2 * sub["n"]).sum() / total_n)
            mae = (sub["MAE"] * sub["n"]).sum() / total_n
        else:
            rmse = sub["RMSE"].mean()
            mae = sub["MAE"].mean()

        rows.append({"target": target, "model": model_name, "RMSE": rmse, "MAE": mae})
    return pd.DataFrame(rows)

def load_metrics(file_map: dict, results_dir: Path) -> pd.DataFrame:
    dfs = []
    for model_name, rel_path in file_map.items():
        p = results_dir / rel_path
        if not p.exists():
            print(f"[WARNING] File not found: {rel_path}")
            continue

        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]

        # Standardize / aggregate
        df = _aggregate_if_per_location(df, model_name)
        df["model"] = model_name
        df["Horizon"] = df["target"].apply(normalize_horizon)

        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)
    out["Horizon"] = pd.Categorical(out["Horizon"], categories=HORIZON_ORDER, ordered=True)
    return out

def plot_metric_sorted_with_average(df: pd.DataFrame, metric: str = "RMSE"):
    """3 horizons + 1 average subplot; bars are sorted worst->best within each panel."""
    panels = HORIZON_ORDER + ["Average"]
    fig, axes = plt.subplots(1, 4, figsize=(26, 10), sharey=False)
    fig.suptitle(f"Model Comparison: {metric}", fontsize=24, weight="bold")

    def _colors(models):
        return ["#EDF5FA" if "Baseline" in m else "#9DC3E6" for m in models]

    for i, panel in enumerate(panels):
        ax = axes[i]

        data = (df[df["Horizon"] == panel]
                      .sort_values(metric, ascending=False))

        if data.empty:
            ax.set_axis_off()
            continue

        sns.barplot(
            data=data, x="model", y=metric, hue="model",
            palette=_colors(data["model"]),
            legend=False, ax=ax
        )
        ax.set_title(f"Forecast Horizon: {panel}", fontsize=18, weight="bold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45, labelsize=14)
        ax.tick_params(axis="y", labelsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.set_ylabel(f"{metric} (mm)" if i == 0 else "")

        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", padding=3, fontsize=12, weight="bold")

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)
    plt.show()

def summary_table_rmse(df: pd.DataFrame) -> pd.DataFrame:
    pivot = df.pivot(index="model", columns="Horizon", values="RMSE")
    pivot = pivot.reindex(columns=HORIZON_ORDER)
    pivot["Average"] = pivot.mean(axis=1)
    return pivot.sort_values(by="1 Day", ascending=True)

def plot_average_bar(pivot: pd.DataFrame, metric_name: str = "RMSE"):
    data = pivot[["Average"]].reset_index().sort_values("Average", ascending=False)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=data, x="model", y="Average", color="#9DC3E6")
    plt.title(f"Average {metric_name} Across Horizons", fontsize=16, weight="bold")
    plt.xlabel("")
    plt.ylabel(f"{metric_name} (mm)")
    plt.xticks(rotation=45, ha="right")
    for container in plt.gca().containers:
        plt.gca().bar_label(container, fmt="%.3f", padding=3, fontsize=11, weight="bold")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

def compare_models_by_horizon(df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(title, fontsize=20, weight="bold")

    for ax, metric, ylabel in [(axes[0], "RMSE", "RMSE (mm)"), (axes[1], "MAE", "MAE (mm)")]:
        sns.barplot(
            data=df, x="Horizon", y=metric, hue="model",
            palette=sns.color_palette("Blues", n_colors=df["model"].nunique()),
            ax=ax
        )
        ax.set_title(metric, fontsize=16, weight="bold")
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_xlabel("")
        ax.tick_params(labelsize=12)
        ax.legend(title="Model Version")
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        for c in ax.containers:
            ax.bar_label(c, fmt="%.3f", padding=3, fontsize=11, weight="bold")

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)
    plt.show()

def improvement_from_avg(avg_df: pd.DataFrame, baseline="LSTM", tuned="Tuned_LSTM"):
    models = set(avg_df["Model Version"])
    if baseline not in models or tuned not in models:
        print(f"[INFO] Improvement calc skipped (need '{baseline}' and '{tuned}').")
        return

    base = avg_df.loc[avg_df["Model Version"] == baseline].iloc[0]
    tune = avg_df.loc[avg_df["Model Version"] == tuned].iloc[0]
    rmse_imp = ((base["RMSE"] - tune["RMSE"]) / base["RMSE"]) * 100
    mae_imp  = ((base["MAE"]  - tune["MAE"])  / base["MAE"])  * 100

    print("\n=== Overall Improvement Analysis ===")
    print(f"Average RMSE Improvement: {rmse_imp:.2f}%")
    print(f"Average MAE Improvement:  {mae_imp:.2f}%")


In [ ]:
# ==========================================
# Load results
# ==========================================
results_df = load_metrics(MODEL_FILES, RESULTS_DIR)

if results_df.empty:
    print("No data loaded.")
else:
    print("Data loaded successfully.")
    display(results_df.head())


In [ ]:
# ==========================================
# Plots (per horizon + average) + numeric summary
# ==========================================
if not results_df.empty:
    # Cell 4 equivalent (+ Average panel)
    plot_metric_sorted_with_average(results_df, metric="RMSE")
    plot_metric_sorted_with_average(results_df, metric="MAE")

    # Cell 5 equivalent (+ Average column + Average plot)
    print("\n=== Numeric Results (RMSE) ===")
    pivot_table = summary_table_rmse(results_df)
    display(pivot_table.round(4))

    plot_average_bar(pivot_table, metric_name="RMSE")

    print("\n=== Performance Analysis ===")
    overall_perf = results_df.groupby("model")["RMSE"].mean().sort_values()
    best_overall = overall_perf.index[0]
    print(f"🏆 BEST MODEL OVERALL: {best_overall} (Avg RMSE: {overall_perf.iloc[0]:.3f})")

    for horizon in HORIZON_ORDER:
        sub = results_df[results_df["Horizon"] == horizon]
        if not sub.empty:
            best_row = sub.loc[sub["RMSE"].idxmin()]
            print(f"   • Best for {horizon}: {best_row['model']} (RMSE: {best_row['RMSE']:.3f})")


In [ ]:
# ==========================================
# Permutation feature importance (LSTM)
# ==========================================
features_path = RESULTS_DIR / "lstm/features.txt"

if not features_path.exists():
    print(f"[WARNING] File not found: {features_path}")
else:
    raw_text = features_path.read_text()

    data = []
    for line in raw_text.strip().split("\n"):
        parts = line.split()
        if len(parts) != 2:
            continue
        try:
            data.append({"Feature": parts[0], "Delta_RMSE_mm": float(parts[1])})
        except ValueError:
            pass

    df = pd.DataFrame(data)
    if df.empty:
        print("[WARNING] No feature importance data parsed.")
    else:
        end_color = "#9DC3E6"   # lighter
        start_color = "#003366" # darker
        cmap = mcolors.LinearSegmentedColormap.from_list("dark_to_medium_blue", [end_color, start_color])
        norm = plt.Normalize(df["Delta_RMSE_mm"].min(), df["Delta_RMSE_mm"].max())
        bar_colors = [cmap(norm(v)) for v in df["Delta_RMSE_mm"]]

        plt.figure(figsize=(10, 8))
        ax = sns.barplot(
            data=df, x="Delta_RMSE_mm", y="Feature",
            palette=bar_colors, hue="Feature", legend=False
        )
        plt.title("Permutation Feature Importance", fontsize=14, fontweight="bold")
        plt.xlabel("Increase in RMSE (mm)", fontsize=12)
        plt.ylabel("Feature", fontsize=12)
        plt.axvline(0, color="#333333", linewidth=1.2)
        plt.tight_layout()
        plt.show()


In [ ]:
# ==========================================
# Best models: LSTM vs Tuned_LSTM (per horizon) + improvement table
# ==========================================
best_results_df = load_metrics(BEST_MODELS, RESULTS_DIR)

if best_results_df.empty:
    print("No comparison data found. Please check file paths in BEST_MODELS.")
else:
    compare_models_by_horizon(best_results_df, "Hyperparameter Tuning Impact: LSTM vs Tuned LSTM")

    print("\n=== Improvement Analysis ===")
    pivot = best_results_df.pivot(index="Horizon", columns="model", values="RMSE")
    if "LSTM" in pivot.columns and "Tuned_LSTM" in pivot.columns:
        pivot["Improvement (%)"] = ((pivot["LSTM"] - pivot["Tuned_LSTM"]) / pivot["LSTM"]) * 100
        print(pivot.round(4))
    else:
        print("Pivot table created but could not calculate improvement (check model names):")
        print(pivot)

    # Overall averages (across horizons)
    avg_results = (best_results_df
                   .groupby("model", as_index=False)[["RMSE", "MAE"]]
                   .mean()
                   .rename(columns={"model": "Model Version"}))

    print("\n=== Average Performance (Across All Horizons) ===")
    display(avg_results.round(4))

    # Average plots
    palette = sns.color_palette("Blues", n_colors=len(avg_results))
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Overall Model Efficiency: Tuned vs Baseline (Average across all horizons)", fontsize=18, weight="bold")

    sns.barplot(data=avg_results, x="Model Version", y="RMSE", palette=palette, ax=axes[0])
    axes[0].set_title("Average RMSE", fontsize=14, weight="bold")
    axes[0].set_xlabel(""); axes[0].set_ylabel("RMSE (mm)")
    axes[0].grid(axis="y", linestyle="--", alpha=0.5)
    for c in axes[0].containers:
        axes[0].bar_label(c, fmt="%.3f", padding=3, fontsize=12, weight="bold")

    sns.barplot(data=avg_results, x="Model Version", y="MAE", palette=palette, ax=axes[1])
    axes[1].set_title("Average MAE", fontsize=14, weight="bold")
    axes[1].set_xlabel(""); axes[1].set_ylabel("MAE (mm)")
    axes[1].grid(axis="y", linestyle="--", alpha=0.5)
    for c in axes[1].containers:
        axes[1].bar_label(c, fmt="%.3f", padding=3, fontsize=12, weight="bold")

    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.show()

    improvement_from_avg(avg_results, baseline="LSTM", tuned="Tuned_LSTM")
